# TRADING BOT - ENTRAÎNEMENT OPTIMISÉ

## Version Améliorée avec Anti-Overfitting

Ce notebook implémente toutes les améliorations identifiées:

1. **Reward Multi-Objectif** - Risk-adjusted, pas juste PnL
2. **Coûts Réalistes** - Spread + Slippage + Commission
3. **Risk Management** - Stop-loss, max drawdown
4. **Anti-Overfitting** - Data augmentation, early stopping
5. **Features Robustes** - Stationnaires et normalisées

---
**IMPORTANT:** Runtime → Change runtime type → GPU (T4)

In [ ]:
# Installation des packages
!pip install -q stable-baselines3[extra] gymnasium pandas numpy matplotlib
!pip install -q torch tensorboard scikit-learn ta
print("Packages installés!")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from typing import Dict, Tuple, Optional
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

import gymnasium as gym
from gymnasium import spaces
from sklearn.preprocessing import StandardScaler
import ta

import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Créer les dossiers
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('logs', exist_ok=True)

## 1. Configuration Optimisée

In [ ]:
@dataclass
class TradingConfig:
    """Configuration optimisée pour le trading."""

    # Capital
    initial_balance: float = 10000

    # Coûts RÉALISTES pour Gold
    spread: float = 0.00025          # 0.025% (2.5 pips)
    commission: float = 0.00010      # 0.01% par côté
    slippage_base: float = 0.00010   # 0.01%
    slippage_volatility_mult: float = 0.5
    slippage_news_mult: float = 2.0

    # Risk Management
    max_position: float = 1.0
    stop_loss_pct: float = 0.02      # 2% stop-loss
    take_profit_pct: float = 0.04    # 4% take-profit
    max_daily_drawdown: float = 0.05 # 5% max DD journalier
    max_total_drawdown: float = 0.15 # 15% max DD total

    # Reward Weights
    reward_pnl_weight: float = 1.0
    reward_sharpe_weight: float = 0.5
    reward_drawdown_penalty: float = 2.0
    reward_overtrade_penalty: float = 0.3
    reward_consistency_bonus: float = 0.2

    # Anti-Overfitting
    price_noise: float = 0.0001      # Bruit sur prix
    dropout_prob: float = 0.02       # 2% features dropout

config = TradingConfig()
print("Configuration chargée:")
print(f"  Stop-loss: {config.stop_loss_pct:.1%}")
print(f"  Take-profit: {config.take_profit_pct:.1%}")
print(f"  Max Drawdown: {config.max_total_drawdown:.1%}")
print(f"  Coût total estimé par trade: ~{(config.spread + 2*config.commission + config.slippage_base)*100:.2f}%")

## 2. Chargement et Préparation des Données

In [ ]:
# Upload des données Gold
from google.colab import files
print("Upload ton fichier XAU_15MIN_2019_2024.csv")
uploaded = files.upload()

for filename in uploaded.keys():
    os.rename(filename, f'data/{filename}')
    print(f"Fichier uploadé: data/{filename}")

In [ ]:
# =============================================================================
# DONNÉES ÉCONOMIQUES RÉELLES
# =============================================================================

NFP_DATA = {
    2019: {1: 304, 2: 56, 3: 189, 4: 263, 5: 75, 6: 224, 7: 164, 8: 130, 9: 136, 10: 128, 11: 266, 12: 147},
    2020: {1: 225, 2: 273, 3: -701, 4: -20687, 5: 2509, 6: 4800, 7: 1763, 8: 1371, 9: 661, 10: 638, 11: 245, 12: -140},
    2021: {1: 233, 2: 468, 3: 916, 4: 278, 5: 614, 6: 962, 7: 1091, 8: 366, 9: 312, 10: 546, 11: 249, 12: 510},
    2022: {1: 504, 2: 714, 3: 431, 4: 428, 5: 390, 6: 372, 7: 528, 8: 315, 9: 263, 10: 261, 11: 263, 12: 223},
    2023: {1: 517, 2: 311, 3: 236, 4: 294, 5: 339, 6: 306, 7: 187, 8: 236, 9: 297, 10: 150, 11: 199, 12: 216},
    2024: {1: 353, 2: 275, 3: 315, 4: 165, 5: 272, 6: 206, 7: 114, 8: 142, 9: 254, 10: 227, 11: 256, 12: 212},
}

CPI_DATA = {
    2019: {1: 1.6, 2: 1.5, 3: 1.9, 4: 2.0, 5: 1.8, 6: 1.6, 7: 1.8, 8: 1.7, 9: 1.7, 10: 1.8, 11: 2.1, 12: 2.3},
    2020: {1: 2.5, 2: 2.3, 3: 1.5, 4: 0.3, 5: 0.1, 6: 0.6, 7: 1.0, 8: 1.3, 9: 1.4, 10: 1.2, 11: 1.2, 12: 1.4},
    2021: {1: 1.4, 2: 1.7, 3: 2.6, 4: 4.2, 5: 5.0, 6: 5.4, 7: 5.4, 8: 5.3, 9: 5.4, 10: 6.2, 11: 6.8, 12: 7.0},
    2022: {1: 7.5, 2: 7.9, 3: 8.5, 4: 8.3, 5: 8.6, 6: 9.1, 7: 8.5, 8: 8.3, 9: 8.2, 10: 7.7, 11: 7.1, 12: 6.5},
    2023: {1: 6.4, 2: 6.0, 3: 5.0, 4: 4.9, 5: 4.0, 6: 3.0, 7: 3.2, 8: 3.7, 9: 3.7, 10: 3.2, 11: 3.1, 12: 3.4},
    2024: {1: 3.1, 2: 3.2, 3: 3.5, 4: 3.4, 5: 3.3, 6: 3.0, 7: 2.9, 8: 2.5, 9: 2.4, 10: 2.6, 11: 2.7, 12: 2.9},
}

FOMC_DATES = {
    2019: ['01-30', '03-20', '05-01', '06-19', '07-31', '09-18', '10-30', '12-11'],
    2020: ['01-29', '03-03', '03-15', '04-29', '06-10', '07-29', '09-16', '11-05', '12-16'],
    2021: ['01-27', '03-17', '04-28', '06-16', '07-28', '09-22', '11-03', '12-15'],
    2022: ['01-26', '03-16', '05-04', '06-15', '07-27', '09-21', '11-02', '12-14'],
    2023: ['02-01', '03-22', '05-03', '06-14', '07-26', '09-20', '11-01', '12-13'],
    2024: ['01-31', '03-20', '05-01', '06-12', '07-31', '09-18', '11-07', '12-18'],
}

def get_first_friday(year, month):
    first_day = datetime(year, month, 1)
    days_until_friday = (4 - first_day.weekday()) % 7
    return first_day + timedelta(days=days_until_friday)

def create_calendar():
    events = []
    for year in range(2019, 2025):
        for month in range(1, 13):
            # NFP
            nfp_date = get_first_friday(year, month)
            nfp_val = NFP_DATA.get(year, {}).get(month, 200)
            nfp_prev = NFP_DATA.get(year, {}).get(month-1) if month > 1 else NFP_DATA.get(year-1, {}).get(12, 200)
            events.append({
                'datetime': nfp_date.replace(hour=12, minute=30),
                'event': 'NFP', 'impact': 'HIGH',
                'actual': nfp_val, 'previous': nfp_prev,
                'surprise': (nfp_val - nfp_prev) / abs(nfp_prev) if nfp_prev else 0
            })
            # CPI
            cpi_date = datetime(year, month, 12)
            cpi_val = CPI_DATA.get(year, {}).get(month, 2.0)
            cpi_prev = CPI_DATA.get(year, {}).get(month-1) if month > 1 else CPI_DATA.get(year-1, {}).get(12, 2.0)
            events.append({
                'datetime': cpi_date.replace(hour=12, minute=30),
                'event': 'CPI', 'impact': 'HIGH',
                'actual': cpi_val, 'previous': cpi_prev,
                'surprise': cpi_val - cpi_prev
            })
        # FOMC
        for date_str in FOMC_DATES.get(year, []):
            try:
                m, d = int(date_str.split('-')[0]), int(date_str.split('-')[1])
                events.append({
                    'datetime': datetime(year, m, d, 18, 0),
                    'event': 'FOMC', 'impact': 'HIGH',
                    'actual': 0, 'previous': 0, 'surprise': 0
                })
            except: pass
    return pd.DataFrame(events).sort_values('datetime')

calendar_df = create_calendar()
print(f"Calendar créé: {len(calendar_df)} événements")

In [ ]:
# Charger et préparer les données Gold
gold_df = pd.read_csv('data/XAU_15MIN_2019_2024.csv', parse_dates=['Date'])
gold_df = gold_df.set_index('Date').sort_index()

print(f"Données Gold: {len(gold_df):,} barres")
print(f"Période: {gold_df.index.min()} → {gold_df.index.max()}")

# Fusionner avec calendar
def add_news_features(df, calendar, window=60):
    df = df.copy()
    df['news_event'] = 0
    df['news_impact'] = 0.0
    df['news_surprise'] = 0.0

    for _, ev in calendar.iterrows():
        mask = (df.index >= ev['datetime'] - timedelta(minutes=window)) & \
               (df.index <= ev['datetime'] + timedelta(minutes=window*2))
        df.loc[mask, 'news_event'] = 1
        df.loc[mask, 'news_impact'] = 1.0
        df.loc[mask, 'news_surprise'] = ev['surprise']
    return df

gold_df = add_news_features(gold_df, calendar_df)
print(f"Barres avec news: {gold_df['news_event'].sum():,} ({gold_df['news_event'].mean()*100:.1f}%)")

In [ ]:
# Split chronologique
n = len(gold_df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

df_train = gold_df.iloc[:train_end].copy()
df_val = gold_df.iloc[train_end:val_end].copy()
df_test = gold_df.iloc[val_end:].copy()

print(f"Train: {len(df_train):,} ({df_train.index.min().date()} → {df_train.index.max().date()})")
print(f"Val:   {len(df_val):,} ({df_val.index.min().date()} → {df_val.index.max().date()})")
print(f"Test:  {len(df_test):,} ({df_test.index.min().date()} → {df_test.index.max().date()})")

## 3. Environnement de Trading OPTIMISÉ

In [ ]:
class OptimizedTradingEnv(gym.Env):
    """
    Environnement de trading OPTIMISÉ avec:
    - Reward multi-objectif
    - Coûts réalistes
    - Risk management intégré
    - Anti-overfitting (data augmentation)
    """

    def __init__(self, df: pd.DataFrame, config: TradingConfig, training: bool = True):
        super().__init__()

        self.config = config
        self.training = training
        self.df_original = df.copy()

        # Préparer les features
        self._prepare_features()

        # Spaces
        self.action_space = spaces.Discrete(3)  # HOLD, BUY, SELL
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(self.n_features,), dtype=np.float32
        )

        # Tracking pour reward multi-objectif
        self.returns_history = []
        self.trade_count = 0
        self.win_count = 0

        self.reset()

    def _prepare_features(self):
        """Prépare les features ROBUSTES et STATIONNAIRES."""
        df = self.df_original.copy()

        # Returns (stationnaire)
        df['returns'] = df['Close'].pct_change()
        df['log_returns'] = np.log(df['Close'] / df['Close'].shift(1))

        # RSI normalisé (z-score par rapport à 50)
        df['rsi_raw'] = ta.momentum.RSIIndicator(df['Close'], window=14).rsi()
        df['rsi'] = (df['rsi_raw'] - 50) / 25  # Normalisé autour de 0

        # MACD normalisé par ATR
        macd = ta.trend.MACD(df['Close'])
        df['atr'] = ta.volatility.AverageTrueRange(df['High'], df['Low'], df['Close']).average_true_range()
        df['macd_norm'] = macd.macd() / (df['atr'] + 1e-8)
        df['macd_signal_norm'] = macd.macd_signal() / (df['atr'] + 1e-8)
        df['macd_hist_norm'] = macd.macd_diff() / (df['atr'] + 1e-8)

        # Bollinger position (0-1)
        bb = ta.volatility.BollingerBands(df['Close'])
        df['bb_position'] = (df['Close'] - bb.bollinger_lband()) / (bb.bollinger_hband() - bb.bollinger_lband() + 1e-8)

        # Prix vs MAs (ratio, pas différence absolue)
        df['sma_20'] = df['Close'].rolling(20).mean()
        df['sma_50'] = df['Close'].rolling(50).mean()
        df['price_sma20_ratio'] = (df['Close'] / df['sma_20']) - 1
        df['price_sma50_ratio'] = (df['Close'] / df['sma_50']) - 1

        # Volatilité réalisée (annualisée)
        df['volatility'] = df['returns'].rolling(20).std() * np.sqrt(252 * 24 * 4)
        df['volatility_zscore'] = (df['volatility'] - df['volatility'].rolling(100).mean()) / (df['volatility'].rolling(100).std() + 1e-8)

        # Volume ratio
        df['volume_ratio'] = df['Volume'] / (df['Volume'].rolling(20).mean() + 1e-8)
        df['volume_ratio'] = np.clip(df['volume_ratio'], 0, 5)  # Cap à 5x

        # Momentum multi-timeframe
        df['momentum_5'] = df['Close'].pct_change(5)
        df['momentum_20'] = df['Close'].pct_change(20)

        # Régime de marché (trend vs range)
        df['trend_strength'] = abs(df['sma_20'] - df['sma_50']) / (df['atr'] + 1e-8)

        # News features
        df['news_event'] = df.get('news_event', 0)
        df['news_impact'] = df.get('news_impact', 0)
        df['news_surprise'] = df.get('news_surprise', 0)

        # Sélectionner les features
        self.feature_columns = [
            'returns', 'log_returns',
            'rsi', 'macd_norm', 'macd_signal_norm', 'macd_hist_norm',
            'bb_position', 'price_sma20_ratio', 'price_sma50_ratio',
            'volatility_zscore', 'volume_ratio',
            'momentum_5', 'momentum_20', 'trend_strength',
            'news_event', 'news_impact', 'news_surprise'
        ]

        # Nettoyer
        df = df.dropna()

        # Normaliser
        self.scaler = StandardScaler()
        self.features = self.scaler.fit_transform(df[self.feature_columns])
        self.prices = df['Close'].values
        self.highs = df['High'].values
        self.lows = df['Low'].values
        self.volumes = df['Volume'].values
        self.volatility = df['volatility'].values
        self.news_events = df['news_event'].values
        self.df_clean = df

        # +3 pour position, pnl, drawdown
        self.n_features = len(self.feature_columns) + 3

    def _apply_data_augmentation(self, features):
        """Applique data augmentation pour anti-overfitting."""
        if not self.training:
            return features

        augmented = features.copy()

        # Bruit gaussien
        noise = np.random.normal(0, self.config.price_noise, features.shape)
        augmented += noise

        # Dropout de features (mettre à 0 aléatoirement)
        if np.random.random() < self.config.dropout_prob:
            drop_idx = np.random.randint(0, len(features))
            augmented[drop_idx] = 0

        return augmented

    def _calculate_transaction_cost(self, price, is_news=False):
        """Calcule les coûts de transaction RÉALISTES."""
        # Spread fixe
        cost = self.config.spread

        # Commission
        cost += self.config.commission

        # Slippage (variable selon volatilité et news)
        slippage = self.config.slippage_base

        # Slippage augmenté si haute volatilité
        current_vol = self.volatility[self.current_step] if self.current_step < len(self.volatility) else 0.2
        slippage *= (1 + self.config.slippage_volatility_mult * current_vol)

        # Slippage augmenté pendant les news
        if is_news:
            slippage *= self.config.slippage_news_mult

        cost += slippage

        return cost * price

    def _calculate_reward(self, pnl, action):
        """Calcule le reward MULTI-OBJECTIF."""
        reward = 0

        # 1. PnL normalisé (base)
        pnl_norm = pnl / self.config.initial_balance
        reward += pnl_norm * self.config.reward_pnl_weight

        # 2. Sharpe rolling (si assez de données)
        if len(self.returns_history) >= 20:
            returns = np.array(self.returns_history[-20:])
            if np.std(returns) > 0:
                sharpe = np.mean(returns) / np.std(returns)
                reward += sharpe * self.config.reward_sharpe_weight * 0.1

        # 3. Pénalité drawdown
        current_dd = self._calculate_drawdown()
        if current_dd > 0.05:  # > 5% drawdown
            reward -= current_dd * self.config.reward_drawdown_penalty

        # 4. Pénalité overtrading
        if action != 0:  # Si pas HOLD
            bars_since_last_trade = getattr(self, 'bars_since_trade', 100)
            if bars_since_last_trade < 4:  # Trade trop fréquent (< 1h pour 15min bars)
                reward -= self.config.reward_overtrade_penalty

        # 5. Bonus consistance (wins consécutifs)
        if pnl > 0:
            self.consecutive_wins = getattr(self, 'consecutive_wins', 0) + 1
            if self.consecutive_wins >= 3:
                reward += self.config.reward_consistency_bonus
        else:
            self.consecutive_wins = 0

        return reward

    def _calculate_drawdown(self):
        """Calcule le drawdown actuel."""
        current_value = self.balance + self._get_unrealized_pnl()
        if current_value >= self.peak_value:
            self.peak_value = current_value
            return 0
        return (self.peak_value - current_value) / self.peak_value

    def _get_unrealized_pnl(self):
        """Calcule le PnL non réalisé de la position ouverte."""
        if self.position == 0:
            return 0
        current_price = self.prices[self.current_step]
        if self.position > 0:
            return (current_price - self.entry_price) * self.position_size
        else:
            return (self.entry_price - current_price) * abs(self.position_size)

    def _check_stop_loss_take_profit(self):
        """Vérifie et exécute les stops."""
        if self.position == 0:
            return False, 0

        current_price = self.prices[self.current_step]
        pnl_pct = 0

        if self.position > 0:  # Long
            pnl_pct = (current_price - self.entry_price) / self.entry_price
        else:  # Short
            pnl_pct = (self.entry_price - current_price) / self.entry_price

        # Stop-loss
        if pnl_pct <= -self.config.stop_loss_pct:
            return True, 'stop_loss'

        # Take-profit
        if pnl_pct >= self.config.take_profit_pct:
            return True, 'take_profit'

        return False, None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.current_step = 0
        self.balance = self.config.initial_balance
        self.position = 0
        self.position_size = 0
        self.entry_price = 0
        self.total_pnl = 0
        self.peak_value = self.config.initial_balance
        self.returns_history = []
        self.trade_count = 0
        self.win_count = 0
        self.bars_since_trade = 100
        self.consecutive_wins = 0
        self.daily_pnl = 0
        self.last_day = None

        return self._get_observation(), {}

    def _get_observation(self):
        features = self.features[self.current_step].copy()

        # Data augmentation si training
        features = self._apply_data_augmentation(features)

        # Ajouter état du portfolio
        position_feat = self.position
        pnl_feat = self.total_pnl / self.config.initial_balance
        dd_feat = self._calculate_drawdown()

        return np.concatenate([features, [position_feat, pnl_feat, dd_feat]]).astype(np.float32)

    def step(self, action):
        current_price = self.prices[self.current_step]
        is_news = self.news_events[self.current_step] > 0
        reward = 0
        pnl = 0

        # Vérifier stop-loss / take-profit
        triggered, trigger_type = self._check_stop_loss_take_profit()
        if triggered:
            pnl = self._close_position(current_price, is_news)
            if trigger_type == 'stop_loss':
                reward -= 0.1  # Pénalité supplémentaire pour stop-loss

        # Vérifier max drawdown journalier
        current_dd = self._calculate_drawdown()
        if current_dd >= self.config.max_total_drawdown:
            # Force close et stop trading
            if self.position != 0:
                pnl += self._close_position(current_price, is_news)
            reward -= 1.0  # Grosse pénalité
            done = True
            self.current_step = len(self.prices) - 1
        else:
            # Exécuter l'action
            if action == 1:  # BUY
                if self.position <= 0:
                    if self.position < 0:
                        pnl = self._close_position(current_price, is_news)
                    self._open_position(1, current_price, is_news)

            elif action == 2:  # SELL
                if self.position >= 0:
                    if self.position > 0:
                        pnl = self._close_position(current_price, is_news)
                    self._open_position(-1, current_price, is_news)

            # Calculer reward
            reward = self._calculate_reward(pnl, action)

            # Avancer
            self.current_step += 1
            self.bars_since_trade += 1

        # Tracker returns pour Sharpe
        self.returns_history.append(pnl / self.config.initial_balance if self.config.initial_balance > 0 else 0)

        # Vérifier si terminé
        done = self.current_step >= len(self.prices) - 1

        # Fermer position si terminé
        if done and self.position != 0:
            final_pnl = self._close_position(self.prices[self.current_step], False)
            reward += self._calculate_reward(final_pnl, 0)

        info = {
            'total_pnl': self.total_pnl,
            'position': self.position,
            'drawdown': self._calculate_drawdown(),
            'trade_count': self.trade_count,
            'win_rate': self.win_count / max(1, self.trade_count)
        }

        return self._get_observation(), reward, done, False, info

    def _open_position(self, direction, price, is_news):
        """Ouvre une position."""
        cost = self._calculate_transaction_cost(price, is_news)
        self.position = direction
        self.position_size = self.config.max_position
        self.entry_price = price
        self.balance -= cost
        self.bars_since_trade = 0

    def _close_position(self, price, is_news):
        """Ferme la position et retourne le PnL."""
        if self.position == 0:
            return 0

        cost = self._calculate_transaction_cost(price, is_news)

        if self.position > 0:
            pnl = (price - self.entry_price) * self.position_size - cost
        else:
            pnl = (self.entry_price - price) * abs(self.position_size) - cost

        self.total_pnl += pnl
        self.balance += pnl
        self.trade_count += 1
        if pnl > 0:
            self.win_count += 1

        self.position = 0
        self.position_size = 0
        self.entry_price = 0

        return pnl

# Test
env = OptimizedTradingEnv(df_train, config, training=True)
obs, _ = env.reset()
print(f"Environnement créé!")
print(f"  Observation shape: {obs.shape}")
print(f"  Features: {env.n_features}")

## 4. Entraînement avec Early Stopping

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback

class OptimizedCallback(BaseCallback):
    """Callback avec early stopping basé sur validation."""

    def __init__(self, eval_env, eval_freq=10000, patience=5, min_improvement=0.01):
        super().__init__()
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.patience = patience
        self.min_improvement = min_improvement

        self.best_sharpe = -np.inf
        self.no_improvement_count = 0
        self.results = []

    def _on_step(self):
        if self.n_calls % self.eval_freq == 0:
            # Évaluer sur validation
            metrics = self._evaluate()

            self.results.append({
                'timestep': self.n_calls,
                **metrics
            })

            print(f"Step {self.n_calls:,}: Sharpe={metrics['sharpe']:.2f}, "
                  f"MaxDD={metrics['max_dd']:.1%}, WinRate={metrics['win_rate']:.1%}")

            # Early stopping check
            if metrics['sharpe'] > self.best_sharpe + self.min_improvement:
                self.best_sharpe = metrics['sharpe']
                self.no_improvement_count = 0
                self.model.save('models/best_model')
                print(f"  → Nouveau meilleur modèle! Sharpe={self.best_sharpe:.2f}")
            else:
                self.no_improvement_count += 1
                print(f"  → Pas d'amélioration ({self.no_improvement_count}/{self.patience})")

            if self.no_improvement_count >= self.patience:
                print(f"\n⚠️ Early stopping! Pas d'amélioration depuis {self.patience} évaluations.")
                return False

        return True

    def _evaluate(self):
        """Évalue le modèle sur l'environnement de validation."""
        obs, _ = self.eval_env.reset()
        done = False
        portfolio_values = [config.initial_balance]

        while not done:
            action, _ = self.model.predict(obs, deterministic=True)
            obs, reward, done, _, info = self.eval_env.step(action)
            portfolio_values.append(config.initial_balance + info['total_pnl'])

        # Métriques
        pv = np.array(portfolio_values)
        returns = np.diff(pv) / pv[:-1]

        sharpe = np.mean(returns) / (np.std(returns) + 1e-8) * np.sqrt(252 * 24 * 4)
        peak = np.maximum.accumulate(pv)
        max_dd = np.max((peak - pv) / peak)

        return {
            'sharpe': sharpe,
            'max_dd': max_dd,
            'total_return': (pv[-1] / pv[0]) - 1,
            'win_rate': info.get('win_rate', 0),
            'trade_count': info.get('trade_count', 0)
        }

# Hyperparamètres OPTIMISÉS
HYPERPARAMS = {
    'learning_rate': 3e-5,
    'n_steps': 2048,
    'batch_size': 128,
    'n_epochs': 10,
    'gamma': 0.99,
    'gae_lambda': 0.95,
    'clip_range': 0.2,
    'ent_coef': 0.05,  # Exploration importante
    'vf_coef': 0.5,
    'max_grad_norm': 0.5,
}

TOTAL_TIMESTEPS = 500_000  # Peut être ajusté
EVAL_FREQ = 10_000
PATIENCE = 5  # Early stopping après 5 évals sans amélioration

print(f"Configuration:")
print(f"  Total timesteps: {TOTAL_TIMESTEPS:,}")
print(f"  Eval frequency: {EVAL_FREQ:,}")
print(f"  Early stopping patience: {PATIENCE}")

In [ ]:
# Créer les environnements
train_env = OptimizedTradingEnv(df_train, config, training=True)
eval_env = OptimizedTradingEnv(df_val, config, training=False)

# Callback
callback = OptimizedCallback(
    eval_env=eval_env,
    eval_freq=EVAL_FREQ,
    patience=PATIENCE
)

# Créer le modèle
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = PPO(
    'MlpPolicy',
    train_env,
    verbose=1,
    tensorboard_log='logs/',
    device=device,
    **HYPERPARAMS
)

print(f"\nDémarrage de l'entraînement sur {device.upper()}...")
print("="*60)

In [ ]:
# Entraînement
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=callback,
    progress_bar=True
)

# Sauvegarder le modèle final
model.save('models/final_model')
print(f"\nEntraînement terminé!")
print(f"Meilleur Sharpe (validation): {callback.best_sharpe:.2f}")

## 5. Évaluation Finale sur Test

In [ ]:
# Charger le meilleur modèle
best_model = PPO.load('models/best_model')

# Évaluer sur TEST (données jamais vues)
test_env = OptimizedTradingEnv(df_test, config, training=False)
obs, _ = test_env.reset()

done = False
portfolio_values = [config.initial_balance]
actions = []
positions = [0]

while not done:
    action, _ = best_model.predict(obs, deterministic=True)
    obs, reward, done, _, info = test_env.step(action)

    portfolio_values.append(config.initial_balance + info['total_pnl'])
    actions.append(action)
    positions.append(info['position'])

# Calculer les métriques
pv = np.array(portfolio_values)
returns = np.diff(pv) / pv[:-1]

# Sharpe
sharpe = np.mean(returns) / (np.std(returns) + 1e-8) * np.sqrt(252 * 24 * 4)

# Sortino
downside = returns[returns < 0]
sortino = np.mean(returns) / (np.std(downside) + 1e-8) * np.sqrt(252 * 24 * 4) if len(downside) > 0 else 0

# Max Drawdown
peak = np.maximum.accumulate(pv)
drawdown = (peak - pv) / peak
max_dd = np.max(drawdown)

# Calmar
annual_return = (pv[-1] / pv[0]) ** (252 * 24 * 4 / len(pv)) - 1
calmar = annual_return / max_dd if max_dd > 0 else 0

# Win Rate
win_rate = info['win_rate']

# Cumulative Return
cum_return = (pv[-1] / pv[0]) - 1

print(f"\n{'='*60}")
print(f"RÉSULTATS FINAUX - DONNÉES TEST (JAMAIS VUES)")
print(f"{'='*60}")
print(f"")
print(f"  Sharpe Ratio:      {sharpe:>10.2f}  {'✅' if sharpe >= 1.0 else '⚠️'} (objectif: > 1.0)")
print(f"  Sortino Ratio:     {sortino:>10.2f}  {'✅' if sortino >= 1.5 else '⚠️'} (objectif: > 1.5)")
print(f"  Calmar Ratio:      {calmar:>10.2f}  {'✅' if calmar >= 1.0 else '⚠️'} (objectif: > 1.0)")
print(f"  Max Drawdown:      {max_dd:>10.1%}  {'✅' if max_dd <= 0.15 else '❌'} (objectif: < 15%)")
print(f"  Win Rate:          {win_rate:>10.1%}  {'✅' if win_rate >= 0.50 else '⚠️'} (objectif: > 50%)")
print(f"  Cumulative Return: {cum_return:>10.1%}")
print(f"")
print(f"  Capital initial:   ${pv[0]:>10,.0f}")
print(f"  Capital final:     ${pv[-1]:>10,.0f}")
print(f"  Profit/Perte:      ${pv[-1]-pv[0]:>+10,.0f}")
print(f"  Nombre de trades:  {info['trade_count']:>10}")
print(f"{'='*60}")

# Verdict
if sharpe >= 1.0 and max_dd <= 0.15 and win_rate >= 0.45:
    print(f"\n✅ BOT PRÊT POUR PAPER TRADING!")
    print(f"   Prochaine étape: 4 semaines de paper trading sur MT5 démo")
else:
    print(f"\n⚠️ Bot pas encore prêt. Suggestions:")
    if sharpe < 1.0:
        print(f"   - Sharpe trop bas: augmenter le training ou ajuster les rewards")
    if max_dd > 0.15:
        print(f"   - Drawdown trop élevé: renforcer le risk management")
    if win_rate < 0.45:
        print(f"   - Win rate faible: revoir la stratégie ou les features")

In [ ]:
# Visualisations
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# 1. Portfolio
axes[0].plot(pv, 'b-', linewidth=0.8)
axes[0].axhline(y=config.initial_balance, color='gray', linestyle='--')
axes[0].fill_between(range(len(pv)), pv, config.initial_balance,
                     where=pv >= config.initial_balance, alpha=0.3, color='green')
axes[0].fill_between(range(len(pv)), pv, config.initial_balance,
                     where=pv < config.initial_balance, alpha=0.3, color='red')
axes[0].set_title(f'Portfolio Value (Sharpe: {sharpe:.2f}, Return: {cum_return:.1%})')
axes[0].set_ylabel('Value ($)')
axes[0].grid(True, alpha=0.3)

# 2. Drawdown
axes[1].fill_between(range(len(drawdown)), -drawdown*100, 0, alpha=0.7, color='red')
axes[1].axhline(y=-config.max_total_drawdown*100, color='darkred', linestyle='--', label=f'Max DD Limit ({config.max_total_drawdown:.0%})')
axes[1].set_title(f'Drawdown (Max: {max_dd:.1%})')
axes[1].set_ylabel('Drawdown (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Actions
action_counts = pd.Series(actions).value_counts().sort_index()
labels = ['HOLD', 'BUY', 'SELL']
colors = ['gray', 'green', 'red']
bars = axes[2].bar([labels[i] for i in action_counts.index],
                   action_counts.values,
                   color=[colors[i] for i in action_counts.index])
axes[2].set_title('Distribution des Actions')
axes[2].set_ylabel('Count')
for bar, count in zip(bars, action_counts.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'{count}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('results_test.png', dpi=150)
plt.show()

In [ ]:
# Télécharger le modèle
!zip -r trading_bot_model.zip models/
files.download('trading_bot_model.zip')
files.download('results_test.png')
print("Fichiers téléchargés!")